# FireBIM — 벽·창·문 모델(STR) GPU 학습 (Google Colab)

**먼저: 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU` 선택 후 저장.** 그다음 위에서부터 셀을 차례로 실행(▶)합니다.

- 준비물(Google Drive `내 드라이브/빔프로젝트/`): `firebim/` 저장소 폴더(또는 `firebim_github.zip`), `239.건축 도면 데이터/…/TS_STR_1.zip·TS_STR_2.zip·TL_STR.zip`, (선택) `_tools/syn_cache_1.zip`·`syn_cache_2.zip`
- 결과는 `빔프로젝트/colab_results/` 에 저장됩니다(체크포인트·ONNX 모델·평가표). 연결이 끊겨도 체크포인트에서 이어 학습 가능.
- ※ AI허브 자료는 이 Colab(본인 계정) 안에서만 사용 — 외부 공개·재배포 금지. 실제 소방활동 정보카드는 올리지 마세요.


In [ ]:
# 1) Drive 연결 + GPU 확인
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
BASE = '/content/drive/MyDrive/빔프로젝트'
DATA = BASE + '/239.건축 도면 데이터/01-1.정식개방데이터'
OUT = BASE + '/colab_results'; os.makedirs(OUT, exist_ok=True)
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout or 'GPU 없음 → 런타임 유형을 T4 GPU로 바꾸세요')
print(os.listdir(BASE))

In [ ]:
# 2) 코드 풀기 + 패키지
import zipfile, shutil
if os.path.exists('/content/firebim'): shutil.rmtree('/content/firebim')
if os.path.exists(BASE + '/firebim/firebim'):                      # Drive 의 저장소 폴더(GitHub 용과 같은 폴더)
    shutil.copytree(BASE + '/firebim', '/content/firebim', ignore=shutil.ignore_patterns('.git'))
else:
    zipfile.ZipFile(BASE + '/firebim_github.zip').extractall('/content')
%cd /content/firebim
!pip -q install onnx onnxruntime opencv-python-headless shapely
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

In [ ]:
# 3) 라벨 풀기 + 색인 (수 분)
AI = '/content/aihub'; os.makedirs(AI + '/labels', exist_ok=True)
for z in ['TL_STR']:
    zipfile.ZipFile(f'{DATA}/Training/02.라벨링데이터/{z}.zip').extractall(f'{AI}/labels/{z}')
!python tools/aihub/build_index.py /content/aihub

In [ ]:
# 4) STR 원천 평면도(FP) 전부 → 긴 변 2048 축소 PNG + meta.json  (약 20~40분, 끊기면 다시 실행하면 이어서 함)
import io, json, numpy as np, cv2
from concurrent.futures import ThreadPoolExecutor
IMG = AI + '/img/STR'; os.makedirs(IMG, exist_ok=True)
mp = IMG + '/meta.json'; meta = json.load(open(mp)) if os.path.exists(mp) else {}
def one(z, n):
    b = os.path.basename(n)
    buf = np.frombuffer(z.read(n), np.uint8)
    g = cv2.imdecode(buf, cv2.IMREAD_REDUCED_GRAYSCALE_2)          # 절반 크기로 빠르게 해독
    H2, W2 = g.shape; W, H = W2 * 2, H2 * 2                          # 원본 크기(근사, ±1px)
    f = 2048 / max(W, H)
    g = cv2.resize(g, (round(W * f), round(H * f)), interpolation=cv2.INTER_AREA)
    cv2.imwrite(os.path.join(IMG, b), g)
    return b, dict(w=W, h=H, f=f)
for part in ['TS_STR_1', 'TS_STR_2']:
    z = zipfile.ZipFile(f'{DATA}/Training/01.원천데이터/{part}.zip')
    names = [n for n in z.namelist() if '_FP_' in n and n.upper().endswith('.PNG') and os.path.basename(n) not in meta]
    print(part, '남은', len(names))
    for i in range(0, len(names), 64):
        with ThreadPoolExecutor(2) as ex:
            for b, m in ex.map(lambda n: one(z, n), names[i:i + 64]): meta[b] = m
        json.dump(meta, open(mp, 'w'))
        if i % 640 == 0: print(part, i, len(meta), flush=True)
print('완료', len(meta))

In [ ]:
# 5) 학습 캐시 만들기 (+ 가상 정보카드 정답 SYN_ 캐시, 있으면)
os.environ['AIHUB'] = AI; os.environ['STR_CACHE'] = '/content/cache/str'
os.makedirs('/content/cache/str', exist_ok=True)
import glob
for syn in sorted(glob.glob(BASE + '/_tools/syn_cache_*.zip')):
    zipfile.ZipFile(syn).extractall('/content/cache/str'); print('SYN 캐시 추가', os.path.basename(syn))
import sys; sys.path.insert(0, '/content/firebim')
from strseg import data as D
from concurrent.futures import ProcessPoolExecutor
todo = [it for it in D.list_images() if not os.path.exists(os.path.join(D.CACHE, it['stem'] + '_l.png'))]
print('캐시할 영상', len(todo))
with ProcessPoolExecutor(2) as ex: list(ex.map(D.cache_one, todo, chunksize=16))
from collections import Counter
print(Counter(D.split_of(i['stem']) for i in D.cached()))

## 6) 학습
- `--width 2.0` = 약 4.4M 파라미터(기존 1.12M의 약 4배), GPU에서 `--minutes 150` 이면 충분.
- 연결이 끊기면 이 셀의 `--warm …` 대신 `--resume {OUT}/ckpt/last.pt` 로 다시 실행.

In [ ]:
!cd /content/firebim && AIHUB=/content/aihub STR_CACHE=/content/cache/str SYN_REPEAT=4 python -m strseg.train \
   --device cuda --width 2.0 --bs 24 --workers 2 --lr 2e-3 --minutes 150 --val_every 10 --nval 150 \
   --out "/content/drive/MyDrive/빔프로젝트/colab_results/ckpt" 2>&1 | grep -E "device|VAL|step [0-9]*00 "


In [ ]:
# 7) ONNX 내보내기 + 보류 674장(이전 보고와 같은 세트) 평가 — 기존 v3 모델과 비교
!cd /content/firebim && python -m strseg.export "{OUT}/ckpt/best.pt" "{OUT}/str_seg_colab.onnx"
!cd /content/firebim && AIHUB=/content/aihub python -m evalr.str_eval pred model --onnx "{OUT}/str_seg_colab.onnx" --out /content/ev_new --subset evalr/held674.json 2>&1 | tail -1
!cd /content/firebim && AIHUB=/content/aihub python -m evalr.str_eval pred model --onnx models/str_seg.onnx --out /content/ev_old --subset evalr/held674.json 2>&1 | tail -1
!cd /content/firebim && python -m evalr.str_eval score model model_vec --out /content/ev_old | head -4
!cd /content/firebim && python -m evalr.str_eval score model model_vec --out /content/ev_new | head -4
!cp /content/ev_new/str_eval.json "{OUT}/str_eval_new.json"; cp /content/ev_old/str_eval.json "{OUT}/str_eval_old.json"
print('결과 저장:', OUT)